# DeepFashion & DeepFashion2 Training Pipeline

This notebook provides a complete pipeline to download, preprocess, fine-tune, and evaluate the YOLOv8 and ResNet50 models on the DeepFashion and DeepFashion2 datasets.

> **Recommended execution environment**: Kaggle Notebooks with **GPU Accelerator (P100 or T4x2)** enabled.

### Prerequisites on Kaggle
1. In the right panel, turn on **Internet access**.
2. Attach your Kaggle API key (`kaggle.json`) or upload it if running locally.

In [ ]:
!pip install -q ultralytics pyyaml faiss-gpu gdown
!pip install -q kaggle

## 1. Setup Data Directories

In [ ]:
import os

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('models/yolo', exist_ok=True)
os.makedirs('models/resnet', exist_ok=True)
os.makedirs('models/faiss', exist_ok=True)
os.makedirs('training', exist_ok=True)

## 2. Upload/Sync Training Scripts
*In a real setup, you would clone the repository here. Since we are in the notebook, we assume the `training/` folder with our Python scripts is present in the working directory.*

In [ ]:
# Example if cloning from github:
# !git clone https://github.com/your-username/AI-Fashion-Recommender.git repo
# !cp -r repo/ml-service/training ./
# !cp -r repo/ml-service/models ./

## 3. Download Datasets
Downloads DeepFashion (Kaggle) and DeepFashion2 (Google Drive).

In [ ]:
!python training/data_downloader.py --data-dir ./data

## 4. Preprocess Datasets

In [ ]:
# Convert DeepFashion2 JSON to YOLO format
!python training/preprocess_deepfashion2.py --input ./data/deepfashion2 --output ./data/deepfashion2_yolo

In [ ]:
# Reorganize DeepFashion for ResNet training
!python training/preprocess_deepfashion.py --input ./data/deepfashion --output ./data/deepfashion_processed

## 5. Fine-Tune YOLOv8s on DeepFashion2
This takes ~5 hours on a T4 GPU.

In [ ]:
!python training/train_yolo.py --data ./data/deepfashion2_yolo/data.yaml --output ./models/yolo --epochs 100 --batch 32

## 6. Fine-Tune ResNet50 on DeepFashion
This takes ~3 hours on a T4 GPU.

In [ ]:
!python training/train_resnet.py --data-dir ./data/deepfashion_processed --output ./models/resnet --epochs 50 --batch 64

## 7. Build FAISS Recommendation Index
Extract features from the In-Shop dataset and build the vector search index.

In [ ]:
!python training/build_recommendation_index.py --images-dir ./data/deepfashion/img --output ./models/faiss

## 8. Evaluate Models
Run full evaluations and generate metrics report.

In [ ]:
!python training/evaluate_models.py --yolo-weights ./models/yolo/best.pt --yolo-data ./data/deepfashion2_yolo/data.yaml --resnet-weights ./models/resnet/fashion_resnet50.pth --resnet-val-dir ./data/deepfashion_processed/val --faiss-index ./models/faiss/fashion.index --output ./models/evaluation_report.json

In [ ]:
import ast, pandas as pd, json, IPython.display
IPython.display.display(pd.read_json('./models/evaluation_report.json', orient='index'))

## 9. Save Weights
Zip up the weights and metadata so you can download them from the Kaggle Output tab.

In [ ]:
!zip -r fashion_models.zip models/
from IPython.display import FileLink
FileLink(r'fashion_models.zip')